In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
import sys
sys.path.append('../')
from datasets import load_dataset, load_from_disk

sampled = load_from_disk("output/xsum_sampled")


In [23]:
from hydra import compose, initialize
with initialize(version_base=None, config_path="../conf"):
    cfg = compose(
        config_name="config",
        overrides=["detector=bert"]
    )

In [24]:
from utils.dataset import combine_human_ai_texts
from utils.filehandler import FileHandler


human_entries = list(sampled['text'])
file_handler = FileHandler(f"outputs/xsum_llama32_3b_h2l.txt")

machine_entries = file_handler.get_lines()
file_handler.close_file()


detector_dataset = combine_human_ai_texts(human_entries, machine_entries)

File already exists: outputs/xsum_llama32_3b_h2l.txt
Opened file: outputs/xsum_llama32_3b_h2l.txt in mode: r
Retrieved 600 lines from file: outputs/xsum_llama32_3b_h2l.txt
Closed file: outputs/xsum_llama32_3b_h2l.txt


Map: 100%|██████████| 1200/1200 [00:00<00:00, 52782.33 examples/s]


In [25]:
from utils.evaluation.Metrics import Metrics
from utils.detector import load_detector_model

roberta= load_detector_model(cfg)

# metrics = Metrics(pred)
# metrics.save_to_file(
#         get_pilot_results_name(dataset_key, detector_key, generator_key, temperature),
#         detector_key, 
#         dataset_key,
#         generator_key,
#         temperature,
#     )


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 42823.48it/s]

In [26]:
preds = roberta.predict(detector_dataset)

Processing batches: 100%|██████████| 38/38 [00:21<00:00,  1.74it/s]


In [27]:
metrics = Metrics(preds)
print(metrics)

AUROC: 0.3577, TPR@1%FPR: 0.0067, ECE: 0.5648, Brier: 0.5646


In [10]:
from utils.detector import RoBertaDefenseModel


roberta = RoBertaDefenseModel()

INIT


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 47676.02it/s]


In [11]:
preds = roberta.predict(detector_dataset)

Processing batches: 100%|██████████| 75/75 [00:07<00:00,  9.65it/s]


In [12]:
metrics = Metrics(preds)
print(metrics)

AUROC: 0.6014, TPR@1%FPR: 0.0217, ECE: 0.4133, Brier: 0.4184
